# 08 - SQL Data Chunking Hands-On

This notebook chunks only `Documents/SQL Data`.

Current state note:
- if folder is empty, the notebook still generates a quality report with `no_files_found`.

Outputs:
- per-file chunk artifacts: `data/sql_data/chunk_artifacts`
- quality reports: `data/sql_data/quality_reports`


In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib, json, re
import pandas as pd
import tiktoken
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

enc = tiktoken.get_encoding('cl100k_base')
print('Environment ready')


In [ ]:
REPO_ROOT = Path(r"C:/Users/Lenovo/source/repos/RAG_Handson")
SRC_ROOT = REPO_ROOT / "Documents" / "SQL Data"
OUT_CHUNKS = REPO_ROOT / "Handson" / "01-Chunking" / "data" / "sql_data" / "chunk_artifacts"
OUT_REPORTS = REPO_ROOT / "Handson" / "01-Chunking" / "data" / "sql_data" / "quality_reports"
OUT_CHUNKS.mkdir(parents=True, exist_ok=True)
OUT_REPORTS.mkdir(parents=True, exist_ok=True)

ALLOWED_EXT = {'.sql', '.md', '.txt', '.csv'}
files = sorted([p for p in SRC_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in ALLOWED_EXT]) if SRC_ROOT.exists() else []
print('Files discovered:', len(files))
for f in files[:10]:
    print('-', f.relative_to(REPO_ROOT))


In [ ]:
def count_tokens(text: str) -> int:
    return len(enc.encode(text))

def chunk_id(text: str) -> str:
    return 'sql_' + hashlib.md5(text.encode('utf-8')).hexdigest()[:12]

def clean_text(text: str) -> str:
    text = text.replace('\x00', '').replace('ﬁ','fi').replace('ﬂ','fl')
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def split_sql_or_text(text: str, source: str, ext: str) -> list[Document]:
    if ext == '.sql':
        separators = ['\n\n', '\n', ';', ' ', '']
    else:
        separators = ['\n\n', '\n', '. ', ' ', '']
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=900,
        chunk_overlap=120,
        separators=separators
    )
    return splitter.create_documents([text], metadatas=[{'source': source, 'corpus': 'sql_data', 'doc_type': 'sql_data_docs'}])


In [ ]:
all_chunks = []
summary_rows = []

if len(files) == 0:
    summary_rows.append({'source': '(none)', 'chunks': 0, 'max_tokens': 0, 'note': 'no_files_found'})
else:
    for f in files:
        rel = str(f.relative_to(REPO_ROOT))
        docs = TextLoader(str(f), encoding='utf-8').load()
        text = clean_text(docs[0].page_content if docs else '')
        if not text:
            summary_rows.append({'source': rel, 'chunks': 0, 'max_tokens': 0, 'note': 'empty_after_load'})
            continue

        chunks = split_sql_or_text(text, source=rel, ext=f.suffix.lower())
        max_tokens = 0
        for i, c in enumerate(chunks):
            t = count_tokens(c.page_content)
            c.metadata.update({'chunk_index': i, 'chunk_id': chunk_id(c.page_content), 'char_count': len(c.page_content), 'token_count': t, 'file_ext': f.suffix.lower()})
            max_tokens = max(max_tokens, t)
        all_chunks.extend(chunks)
        summary_rows.append({'source': rel, 'chunks': len(chunks), 'max_tokens': max_tokens, 'note': 'ok'})

def save_jsonl(chunks: list[Document], out_file: Path):
    with out_file.open('w', encoding='utf-8') as wf:
        for c in chunks:
            wf.write(json.dumps({'page_content': c.page_content, 'metadata': c.metadata}, ensure_ascii=False) + '\n')

save_jsonl(all_chunks, OUT_CHUNKS / 'all_sql_data_chunks.jsonl')

if all_chunks:
    by_source = {}
    for c in all_chunks:
        by_source.setdefault(c.metadata['source'], []).append(c)
    for src, ch in by_source.items():
        save_jsonl(ch, OUT_CHUNKS / f"{Path(src).stem}_chunks.jsonl")

summary_df = pd.DataFrame(summary_rows)
run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
summary_df.to_csv(OUT_REPORTS / f'sql_data_file_summary_{run_id}.csv', index=False)
(OUT_REPORTS / f'sql_data_file_summary_{run_id}.json').write_text(summary_df.to_json(orient='records', indent=2), encoding='utf-8')

print('Total chunks:', len(all_chunks))
display(summary_df)


In [ ]:
print('='*80)
print('SQL DATA CHUNKING COMPLETE')
print('='*80)
print('Chunk artifacts:', OUT_CHUNKS)
print('Quality reports:', OUT_REPORTS)
